# MWPL-Based Cross-Sectional Stock Screening Model

## Objective

The objective of this study is to test whether MWPL (Market Wide Position Limit) dynamics and Open Interest behaviour contain predictive information about future short-term stock returns.

The analysis focuses on identifying:
- MWPL squeeze behaviour
- Ban-period dynamics
- Open Interest build-up and unwinding
- Momentum and reversal effects
- Market regime conditions using participant positioning

The goal is not to predict absolute market direction, but to rank stocks cross-sectionally based on their probability of outperforming over the next 5 trading days.

---

## Research Hypothesis

Stocks with rapidly increasing MWPL and Open Interest may experience unusual short-term price movements because too many traders become positioned in the same direction. When these crowded positions start getting closed (short covering or long unwinding), prices can move sharply due to derivatives market pressure.

---

## Important Market Structure Insight

Participant-wise OI data published by NSE is market-wide rather than stock-specific.

Therefore:
- Participant positioning is treated as a market regime signal
- Stock ranking is driven primarily by stock-level MWPL and OI features

This avoids introducing misleading stock-level information from market-level participant data.

---

## Methodology

The research pipeline includes:

1. Data cleaning and consolidation
2. Stock-level MWPL and OI feature engineering
3. Market regime construction
4. Cross-sectional ranking framework
5. Purged walk-forward validation
6. Ensemble machine learning models
7. Out-of-sample backtesting

---

## Validation Framework

To avoid look-ahead bias and data leakage:
- Walk-forward validation is used
- Embargo periods are applied between train and test sets
- Models are evaluated fully out-of-sample

---

## Dataset

The dataset consists of:
- NSE MWPL data
- Stock-level price and OI features
- Participant-wise derivatives positioning data

covering approximately 67 trading days across the F&O universe.

# `Upload files and create one Excel file`

In [1]:
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings("ignore")

from google.colab import files

print("Upload all Participant OI files")
uploaded_participant = files.upload()

participant_files = []

for filename in uploaded_participant.keys():
    try:
        m = re.search(r"(\d{2})(\d{2})(\d{4})", filename)
        if not m:
            print("Skipped:", filename)
            continue

        file_date = pd.to_datetime(f"{m.group(3)}-{m.group(2)}-{m.group(1)}")

        df = pd.read_csv(filename, skiprows=1)
        df.columns = (
            df.columns.str.strip()
            .str.replace("\n", " ", regex=False)
            .str.replace(r"\s+", " ", regex=True)
        )

        df = df[df["Client Type"].astype(str).str.upper() != "TOTAL"]
        df["Date"] = file_date

        participant_files.append(df)

    except Exception as e:
        print("Error:", filename, e)

participant_raw = pd.concat(participant_files, ignore_index=True)

participant_raw = (
    participant_raw
    .dropna(subset=["Date", "Client Type"])
    .drop_duplicates(subset=["Date", "Client Type"], keep="last")
)

print("Participant data shape:", participant_raw.shape)
print("Unique dates:", participant_raw["Date"].nunique())

print("Upload MWPL + stock price dataset")
uploaded_mwpl = files.upload()
mwpl_file = list(uploaded_mwpl.keys())[0]

mwpl = pd.read_csv(mwpl_file)
mwpl.columns = (
    mwpl.columns.str.strip()
    .str.replace("\n", " ", regex=False)
    .str.replace(r"\s+", " ", regex=True)
)

mwpl["Date"] = pd.to_datetime(mwpl["Date"]).dt.normalize()

print("MWPL data shape:", mwpl.shape)

with pd.ExcelWriter("combined_mwpl_participant_data.xlsx", engine="openpyxl") as writer:
    mwpl.to_excel(writer, sheet_name="MWPL_Stock_Data", index=False)
    participant_raw.to_excel(writer, sheet_name="Participant_OI_Raw", index=False)

files.download("combined_mwpl_participant_data.xlsx")

Upload all Participant OI files


Saving fao_participant_oi_12052026.csv to fao_participant_oi_12052026.csv
Saving fao_participant_oi_11052026.csv to fao_participant_oi_11052026.csv
Saving fao_participant_oi_08052026.csv to fao_participant_oi_08052026.csv
Saving fao_participant_oi_07052026.csv to fao_participant_oi_07052026.csv
Saving fao_participant_oi_06052026.csv to fao_participant_oi_06052026.csv
Saving fao_participant_oi_05052026.csv to fao_participant_oi_05052026.csv
Saving fao_participant_oi_04052026.csv to fao_participant_oi_04052026.csv
Saving fao_participant_oi_30042026.csv to fao_participant_oi_30042026.csv
Saving fao_participant_oi_28042026.csv to fao_participant_oi_28042026.csv
Saving fao_participant_oi_27042026.csv to fao_participant_oi_27042026.csv
Saving fao_participant_oi_24042026.csv to fao_participant_oi_24042026.csv
Saving fao_participant_oi_23042026.csv to fao_participant_oi_23042026.csv
Saving fao_participant_oi_22042026.csv to fao_participant_oi_22042026.csv
Saving fao_participant_oi_21042026.csv

Saving 01_final_merged_mwpl_price_dataset.csv to 01_final_merged_mwpl_price_dataset.csv
MWPL data shape: (12914, 35)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Load combined Excel

In [2]:
excel_file = "combined_mwpl_participant_data.xlsx"

mwpl = pd.read_excel(excel_file, sheet_name="MWPL_Stock_Data")
participant_raw = pd.read_excel(excel_file, sheet_name="Participant_OI_Raw")

mwpl["Date"] = pd.to_datetime(mwpl["Date"]).dt.normalize()
participant_raw["Date"] = pd.to_datetime(participant_raw["Date"]).dt.normalize()

print("MWPL:", mwpl.shape)
print("Participant:", participant_raw.shape)

MWPL: (12914, 35)
Participant: (264, 16)


# Participant market-regime features

In [3]:
numeric_cols = [c for c in participant_raw.columns if c not in ["Date", "Client Type"]]

for col in numeric_cols:
    participant_raw[col] = pd.to_numeric(participant_raw[col], errors="coerce")

participant_raw["Net_Futures"] = (
    participant_raw.get("Future Index Long", 0)
    + participant_raw.get("Future Stock Long", 0)
    - participant_raw.get("Future Index Short", 0)
    - participant_raw.get("Future Stock Short", 0)
)

participant = participant_raw.pivot_table(
    index="Date",
    columns="Client Type",
    values="Net_Futures",
    aggfunc="last"
).reset_index()

participant.columns = ["Date"] + [f"{c}_NetFut" for c in participant.columns[1:]]

fii_col = next((c for c in participant.columns if "FII" in c), None)
client_col = next((c for c in participant.columns if "Client" in c), None)
dii_col = next((c for c in participant.columns if "DII" in c), None)

if fii_col:
    participant["FII_NetFut_chg"] = participant[fii_col].diff()
    participant["FII_NetFut_z10"] = (
        participant[fii_col] - participant[fii_col].rolling(10).mean()
    ) / (participant[fii_col].rolling(10).std() + 1e-9)

if fii_col and client_col:
    participant["Retail_vs_FII"] = participant[client_col] - participant[fii_col]
    participant["Retail_vs_FII_z10"] = (
        participant["Retail_vs_FII"] - participant["Retail_vs_FII"].rolling(10).mean()
    ) / (participant["Retail_vs_FII"].rolling(10).std() + 1e-9)

if fii_col and dii_col:
    participant["Institutional_Flow"] = participant[fii_col] + participant[dii_col]
    participant["Inst_Flow_z10"] = (
        participant["Institutional_Flow"] - participant["Institutional_Flow"].rolling(10).mean()
    ) / (participant["Institutional_Flow"].rolling(10).std() + 1e-9)

print(participant.head())

        Date  Client_NetFut  DII_NetFut  FII_NetFut  Pro_NetFut  \
0 2026-02-01        2682480    -4257338     1098462      476396   
1 2026-02-02        2664433    -4252030     1119628      467969   
2 2026-02-03        2582212    -4157166     1213592      361362   
3 2026-02-04        2583682    -4146739     1224613      338444   
4 2026-02-05        2615826    -4143212     1188034      339352   

   FII_NetFut_chg  FII_NetFut_z10  Retail_vs_FII  Retail_vs_FII_z10  \
0             NaN             NaN        1584018                NaN   
1         21166.0             NaN        1544805                NaN   
2         93964.0             NaN        1368620                NaN   
3         11021.0             NaN        1359069                NaN   
4        -36579.0             NaN        1427792                NaN   

   Institutional_Flow  Inst_Flow_z10  
0            -3158876            NaN  
1            -3132402            NaN  
2            -2943574            NaN  
3            -

# Stock-level feature engineering

In [4]:
mwpl = mwpl.sort_values(["NSE Symbol", "Date"]).reset_index(drop=True)

def build_stock_features(g):
    g = g.sort_values("Date").copy()

    g["MWPL_vel"] = g["MWPL_%"].diff()
    g["MWPL_accel"] = g["MWPL_vel"].diff()
    g["MWPL_5D_chg"] = g["MWPL_%"].diff(5)

    g["At_70"] = (g["MWPL_%"] >= 70).astype(int)
    g["At_85"] = (g["MWPL_%"] >= 85).astype(int)
    g["At_95"] = (g["MWPL_%"] >= 95).astype(int)

    g["Ban_Entry"] = ((g["MWPL_%"] >= 95) & (g["MWPL_%"].shift(1) < 95)).astype(int)
    g["Ban_Exit"] = ((g["MWPL_%"] < 80) & (g["MWPL_%"].shift(1) >= 80)).astype(int)

    g["OI_vel"] = g["OI_Change_%"].diff()
    g["OI_3D_sum"] = g["OI_Change_%"].rolling(3).sum()
    g["OI_5D_sum"] = g["OI_Change_%"].rolling(5).sum()

    g["Squeeze_Score"] = g["MWPL_%"] * g["OI_5D_sum"].clip(lower=0)

    if "Return_Past_3D" not in g.columns:
        g["Return_Past_3D"] = g["Return_1D"].rolling(3).sum()

    g["MomReversal"] = (-g["Return_Past_3D"] * g["At_85"]).clip(lower=0)

    return g

mwpl = mwpl.groupby("NSE Symbol", group_keys=False).apply(build_stock_features)

print("Feature-engineered MWPL data:", mwpl.shape)

Feature-engineered MWPL data: (12914, 48)


# Market regime + merge

In [5]:
market = (
    mwpl.groupby("Date")
    .agg(
        Mkt_Return_1D=("Return_1D", "mean"),
        Mkt_Breadth=("Return_1D", lambda x: (x > 0).mean()),
        Mkt_Vol=("Realized_Vol_5D", "mean"),
        Mkt_OI_chg=("OI_Change_%", "mean"),
        Mkt_MWPL=("MWPL_%", "mean"),
        Mkt_High_Squeeze=("At_85", "mean")
    )
    .reset_index()
)

market["Mkt_Momentum_3D"] = market["Mkt_Return_1D"].rolling(3).mean()

merged = mwpl.merge(market, on="Date", how="left")
merged = merged.merge(participant, on="Date", how="left")

print("Merged dataset:", merged.shape)

Merged dataset: (12914, 65)


# Cross-sectional ranks + target

In [6]:
rank_cols = [
    "MWPL_%", "MWPL_vel", "MWPL_5D_chg",
    "OI_Change_%", "OI_5D_sum",
    "Squeeze_Score", "Return_Past_3D",
    "Realized_Vol_5D", "Price_Change_%"
]

for col in rank_cols:
    if col in merged.columns:
        merged[f"{col}_rank"] = merged.groupby("Date")[col].rank(pct=True)

merged["FwdReturn_rank"] = merged.groupby("Date")["Return_5D"].rank(pct=True)
merged["Target_Buy"] = (merged["FwdReturn_rank"] >= 0.80).astype(int)

print("Target balance:")
print(merged["Target_Buy"].value_counts(normalize=True))

Target balance:
Target_Buy
0    0.813536
1    0.186464
Name: proportion, dtype: float64


# Model data preparation

In [7]:
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.metrics import roc_auc_score

FEATURES = [
    "MWPL_%", "MWPL_vel", "MWPL_accel", "MWPL_5D_chg",
    "At_70", "At_85", "At_95", "Ban_Entry", "Ban_Exit",
    "OI_Change_%", "OI_vel", "OI_3D_sum", "OI_5D_sum",
    "Squeeze_Score", "Return_Past_3D", "MomReversal",
    "Realized_Vol_5D", "Price_Change_%",
    "MWPL_%_rank", "MWPL_vel_rank", "MWPL_5D_chg_rank",
    "OI_Change_%_rank", "OI_5D_sum_rank",
    "Squeeze_Score_rank", "Return_Past_3D_rank",
    "Realized_Vol_5D_rank", "Price_Change_%_rank",
    "Mkt_Return_1D", "Mkt_Breadth", "Mkt_Vol",
    "Mkt_OI_chg", "Mkt_MWPL", "Mkt_High_Squeeze",
    "Mkt_Momentum_3D",
    "FII_NetFut_chg", "FII_NetFut_z10",
    "Retail_vs_FII_z10", "Inst_Flow_z10"
]

FEATURES = [f for f in FEATURES if f in merged.columns]

ID_COLS = ["Date", "NSE Symbol", "Scrip Name", "Return_5D", "Target_Buy"]

model_data = merged[ID_COLS + FEATURES].replace([np.inf, -np.inf], np.nan).dropna()
model_data = model_data.sort_values("Date").reset_index(drop=True)

print("Model data:", model_data.shape)
print("Features used:", len(FEATURES))
print("Date range:", model_data["Date"].min(), "to", model_data["Date"].max())

Model data: (10146, 43)
Features used: 38
Date range: 2026-02-12 00:00:00 to 2026-04-28 00:00:00


# Walk-forward validation

In [8]:
CFG = {
    "n_splits": 3,
    "embargo_days": 5,
    "top_pct": 0.10,
    "bottom_pct": 0.10
}

dates = model_data["Date"].sort_values().unique()
fold_size = len(dates) // (CFG["n_splits"] + 1)

oof_results = []
fold_aucs = []

for fold in range(CFG["n_splits"]):
    train_end_idx = fold_size * (fold + 1)
    test_start_idx = train_end_idx + 1
    test_end_idx = min(test_start_idx + fold_size, len(dates))

    train_end_date = dates[train_end_idx]
    test_start_date = dates[test_start_idx]
    test_end_date = dates[test_end_idx - 1]

    train_cutoff = train_end_date - pd.Timedelta(days=CFG["embargo_days"])

    train_mask = model_data["Date"] <= train_cutoff
    test_mask = (model_data["Date"] >= test_start_date) & (model_data["Date"] <= test_end_date)

    X_train = model_data.loc[train_mask, FEATURES]
    y_train = model_data.loc[train_mask, "Target_Buy"]

    X_test = model_data.loc[test_mask, FEATURES]
    y_test = model_data.loc[test_mask, "Target_Buy"]

    rf = RandomForestClassifier(
        n_estimators=300,
        max_depth=5,
        min_samples_leaf=20,
        class_weight={0: 1, 1: 2},
        random_state=42,
        n_jobs=-1
    )

    et = ExtraTreesClassifier(
        n_estimators=300,
        max_depth=5,
        min_samples_leaf=20,
        class_weight={0: 1, 1: 2},
        random_state=42,
        n_jobs=-1
    )

    gb = GradientBoostingClassifier(
        n_estimators=120,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    )

    rf.fit(X_train, y_train)
    et.fit(X_train, y_train)
    gb.fit(X_train, y_train)

    probs = (
        0.40 * rf.predict_proba(X_test)[:, 1]
        + 0.35 * et.predict_proba(X_test)[:, 1]
        + 0.25 * gb.predict_proba(X_test)[:, 1]
    )

    auc = roc_auc_score(y_test, probs)
    fold_aucs.append(auc)

    fold_df = model_data.loc[test_mask, ID_COLS].copy()
    fold_df["Buy_Probability"] = probs
    oof_results.append(fold_df)

    print(f"Fold {fold+1}: AUC = {auc:.3f}")

oof = pd.concat(oof_results, ignore_index=True)

print("Mean OOF AUC:", np.mean(fold_aucs))

Fold 1: AUC = 0.528
Fold 2: AUC = 0.495
Fold 3: AUC = 0.490
Mean OOF AUC: 0.5040583382940392


# Backtest

In [9]:
def sharpe(x):
    return x.mean() / x.std() if x.std() != 0 else np.nan

def max_drawdown(x):
    cum = (1 + x).cumprod()
    return (cum / cum.cummax() - 1).min()

oof["Daily_Rank"] = oof.groupby("Date")["Buy_Probability"].rank(ascending=False, method="first")
oof["Universe_Count"] = oof.groupby("Date")["NSE Symbol"].transform("count")

oof["Top_10pct"] = oof["Daily_Rank"] <= oof["Universe_Count"] * 0.10
oof["Bottom_10pct"] = oof["Daily_Rank"] > oof["Universe_Count"] * 0.90

daily_top = oof[oof["Top_10pct"]].groupby("Date")["Return_5D"].mean()
daily_bottom = oof[oof["Bottom_10pct"]].groupby("Date")["Return_5D"].mean()
daily_all = oof.groupby("Date")["Return_5D"].mean()

excess = daily_top - daily_all
long_short = daily_top - daily_bottom

summary = pd.DataFrame({
    "Metric": [
        "Mean OOF AUC",
        "Top 10% avg 5D return",
        "Universe avg 5D return",
        "Top 10% excess return",
        "Long-short spread",
        "Top 10% hit rate vs universe",
        "Long-short hit rate",
        "Top 10% Sharpe",
        "Long-short Sharpe",
        "Top 10% max drawdown"
    ],
    "Value": [
        np.mean(fold_aucs),
        daily_top.mean(),
        daily_all.mean(),
        excess.mean(),
        long_short.mean(),
        (excess > 0).mean(),
        (long_short > 0).mean(),
        sharpe(daily_top),
        sharpe(long_short),
        max_drawdown(daily_top)
    ]
})

summary

,Metric,Value
0,Mean OOF AUC,0.504058
1,Top 10% avg 5D return,0.009192
2,Universe avg 5D return,0.007612
3,Top 10% excess return,0.001581
4,Long-short spread,0.004302
5,Top 10% hit rate vs universe,0.666667
6,Long-short hit rate,0.555556
7,Top 10% Sharpe,0.300128
8,Long-short Sharpe,0.261519
9,Top 10% max drawdown,-0.219433


# Final model + live recommendation

In [10]:
X_all = model_data[FEATURES]
y_all = model_data["Target_Buy"]

final_model = RandomForestClassifier(
    n_estimators=500,
    max_depth=5,
    min_samples_leaf=20,
    class_weight={0: 1, 1: 2},
    random_state=42,
    n_jobs=-1
)

final_model.fit(X_all, y_all)

latest_date = model_data["Date"].max()
latest = model_data[model_data["Date"] == latest_date].copy()

latest["Buy_Probability"] = final_model.predict_proba(latest[FEATURES])[:, 1]
latest["Rank"] = latest["Buy_Probability"].rank(ascending=False, method="first")

latest["Recommendation"] = "Avoid"
latest.loc[latest["Rank"] <= 10, "Recommendation"] = "Top Pick"
latest.loc[(latest["Rank"] > 10) & (latest["Rank"] <= 25), "Recommendation"] = "Watchlist"

recommendations = latest.sort_values("Buy_Probability", ascending=False)

cols = [
    "Date", "NSE Symbol", "Scrip Name",
    "Buy_Probability", "Rank", "Recommendation",
    "MWPL_%", "MWPL_vel", "OI_5D_sum",
    "Squeeze_Score", "Return_Past_3D",
    "At_85", "At_95", "Ban_Entry"
]

cols = [c for c in cols if c in recommendations.columns]

recommendations[cols].head(25)

,Date,NSE Symbol,Scrip Name,Buy_Probability,Rank,Recommendation,MWPL_%,MWPL_vel,OI_5D_sum,Squeeze_Score,Return_Past_3D,At_85,At_95,Ban_Entry
10113,2026-04-28,CROMPTON,CROMPT GREA CON ELEC LTD,0.508338,1.0,Top Pick,48.884661,4.087150,-17.249184,0.000000,0.064091,0,0,0
10125,2026-04-28,ADANIENT,ADANI ENTERPRISES LIMITED,0.480814,2.0,Top Pick,57.625834,0.866375,-10.806005,0.000000,0.048870,0,0,0
10064,2026-04-28,SUNPHARMA,SUN PHARMACEUTICAL IND L,0.470361,3.0,Top Pick,24.095317,1.796730,20.610439,496.615059,0.039998,0,0,0
10100,2026-04-28,GLENMARK,GLENMARK PHARMACEUTICALS,0.446805,4.0,Top Pick,70.954908,5.134731,1.042532,73.972771,0.029422,0,0,0
9989,2026-04-28,RBLBANK,RBL BANK LIMITED,0.417003,5.0,Top Pick,66.982273,-4.339391,-23.192750,0.000000,0.026564,0,0,0
10139,2026-04-28,BHEL,BHEL,0.403511,6.0,Top Pick,63.496921,-1.995070,-25.899853,0.000000,0.050681,0,0,0
10070,2026-04-28,IDEA,VODAFONE IDEA LIMITED,0.384172,7.0,Top Pick,53.664013,0.360866,-10.053188,0.000000,0.038622,0,0,0
9942,2026-04-28,AMBER,AMBER ENTERPRISES (I) LTD,0.381753,8.0,Top Pick,36.113731,-2.337033,-55.910940,0.000000,0.051956,0,0,0
10041,2026-04-28,IREDA,INDIAN RENEWABLE ENERGY,0.374363,9.0,Top Pick,35.156218,-3.633430,-30.875189,0.000000,0.003854,0,0,0
9961,2026-04-28,BSE,BSE LIMITED,0.374347,10.0,Top Pick,13.306440,-0.724572,-54.801814,0.000000,0.047098,0,0,0


# Conclusion

## Key Findings

The study tested whether MWPL dynamics and derivatives positioning contain predictive information about future short-term stock returns.

The results suggest:

- MWPL and OI features contain weak but unstable predictive information
- Participant-wise OI data is more useful as a market regime indicator rather than a stock-selection signal
- Squeeze-related features and cross-sectional ranking frameworks appear more informative than raw MWPL levels alone

---

## Model Performance

The out-of-sample walk-forward validation produced:

- Mean OOF AUC ≈ 0.50
- Weak and inconsistent predictive performance across folds

This indicates that:
- the current signal is close to noise,
- and the available sample size is insufficient to establish stable predictive power.

---

## Interpretation

The research suggests that:
- MWPL alone is not a strong standalone alpha signal,
- but derivatives positioning behaviour may still provide useful contextual information when combined with richer market features.

The current framework should therefore be interpreted as:
- a research and screening framework,
- rather than a deployable trading strategy.

---

## Limitations

The primary limitations of the study are:

- Limited historical data (~67 trading days)
- Absence of options-chain variables
- No stock-level participant positioning
- Lack of funding/basis data
- Limited ban-period observations

Financial market signals are highly noisy, and robust validation typically requires much longer historical samples.

---

## Future Improvements

Potential extensions include:

- Expanding MWPL history to multiple years
- Adding options-chain metrics (PCR, IV percentile, max pain)
- Adding delivery percentage and short-selling data
- Incorporating futures basis and funding proxies
- Testing sector-relative and event-driven signals

---

## Final Takeaway

Although the current results do not demonstrate strong predictive power, the research establishes a structured framework for studying MWPL-driven positioning effects using proper out-of-sample validation and market microstructure-aware feature engineering.